# 🌿 AI Crop Disease Data Generator using GAN
## DCGAN on PlantVillage Dataset (Tomato Classes)

**Google Colab Training Notebook**

This notebook trains a Deep Convolutional GAN (DCGAN) on tomato disease images
from the PlantVillage dataset to generate realistic synthetic disease images.

---
**Steps:**
1. Mount Google Drive & install dependencies
2. Load dataset with preprocessing
3. Define DCGAN Generator and Discriminator
4. Train the GAN adversarially
5. Plot training loss curves
6. Generate 50 synthetic crop disease images

## Step 1 — Mount Google Drive and Install Dependencies

In [ ]:
# Mount Google Drive so we can access the PlantVillage dataset
from google.colab import drive
drive.mount('/content/drive')

# Install required packages (already available in Colab, but ensures latest versions)
!pip install -q torch torchvision matplotlib numpy Pillow tqdm

## Step 2 — Imports, Hyperparameters, and Device Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import torchvision.utils as vutils
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

# ---------------------------------------------------------------
# UPDATE DATASET_ROOT to where you uploaded PlantVillage on Drive
# ---------------------------------------------------------------
DATASET_ROOT = '/content/drive/MyDrive/PlantVillage'  # <-- change this

# Hyperparameters (must match the project spec)
IMAGE_SIZE    = 128
BATCH_SIZE    = 64
LATENT_DIM    = 100   # Size of random noise vector
NUM_EPOCHS    = 50
LEARNING_RATE = 0.0002
BETA1         = 0.5   # Adam optimizer first moment decay
FEATURE_MAPS  = 64    # Base feature-map multiplier for G and D

# Output directories
SAVE_DIR  = '/content/outputs/generated_images'
MODEL_DIR = '/content/outputs/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Use GPU if available (strongly recommended — enable in Runtime > Change runtime type)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3 — Dataset Loader

Load only **Tomato** class images from PlantVillage. Images are:
- Resized to **128 × 128**
- Normalised to **[-1, 1]** (compatible with Tanh output of Generator)

In [ ]:
def get_dataloader(dataset_root, crop='Tomato', image_size=128,
                   batch_size=64, num_workers=2):
    """
    Build a DataLoader that serves images from all sub-folders
    whose name contains the 'crop' keyword (e.g. 'Tomato').
    """
    # Image preprocessing pipeline
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        # Normalize from [0,1] to [-1,1] — matches Generator's Tanh output
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    # Load the full PlantVillage dataset (ImageFolder reads class sub-dirs)
    full_dataset = datasets.ImageFolder(root=dataset_root, transform=transform)

    # Filter: keep only images from the chosen crop's classes
    crop_lower = crop.lower()
    filtered_indices = [
        idx for idx, (_, label) in enumerate(full_dataset.samples)
        if crop_lower in full_dataset.classes[label].lower()
    ]

    print(f'Crop: {crop} | Images found: {len(filtered_indices)}')
    print(f'Classes: {[c for c in full_dataset.classes if crop_lower in c.lower()]}')

    crop_dataset = torch.utils.data.Subset(full_dataset, filtered_indices)

    loader = DataLoader(
        crop_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True,        # Keep all batches the same size
        pin_memory=torch.cuda.is_available(),
    )
    print(f'Batches per epoch: {len(loader)}')
    return loader

dataloader = get_dataloader(DATASET_ROOT, batch_size=BATCH_SIZE)

# Visualize a few real images
real_batch, _ = next(iter(dataloader))
grid = vutils.make_grid(real_batch[:32], nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title('Sample Real Tomato Disease Images from PlantVillage')
plt.axis('off')
plt.show()

## Step 4 — DCGAN Generator

Architecture: **Noise (100-d) → Linear → ConvTranspose2d blocks → 3×128×128 image**

In [ ]:
class Generator(nn.Module):
    """
    DCGAN Generator
    Input:  random noise vector z of shape (batch, latent_dim)
    Output: RGB image of shape (batch, 3, 128, 128) in range [-1, 1]
    """
    def __init__(self, latent_dim=100, fm=64):
        super(Generator, self).__init__()

        # Project noise to a small 4x4 feature map
        self.project = nn.Sequential(
            nn.Linear(latent_dim, 512 * 4 * 4, bias=False),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(inplace=True),
        )

        # Transposed convolutions: progressively double spatial size
        self.main = nn.Sequential(
            # 4x4 -> 8x8
            nn.ConvTranspose2d(512, fm * 8, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.ReLU(inplace=True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(fm * 8, fm * 4, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 4),
            nn.ReLU(inplace=True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(fm * 4, fm * 2, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.ReLU(inplace=True),
            # 32x32 -> 64x64
            nn.ConvTranspose2d(fm * 2, fm, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm),
            nn.ReLU(inplace=True),
            # 64x64 -> 128x128 (output layer)
            nn.ConvTranspose2d(fm, 3, 4, stride=2, padding=1, bias=False),
            nn.Tanh(),  # Output in [-1, 1]
        )
        self._init_weights()

    def forward(self, z):
        x = self.project(z)                   # (B, 512*4*4)
        x = x.view(x.size(0), 512, 4, 4)     # (B, 512, 4, 4)
        return self.main(x)                   # (B, 3, 128, 128)

    def _init_weights(self):
        """DCGAN paper: Conv weights ~ N(0, 0.02), BatchNorm ~ N(1, 0.02)"""
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d)):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight.data, 1.0, 0.02)
                nn.init.constant_(m.bias.data, 0)

# Sanity check
G = Generator(LATENT_DIM, FEATURE_MAPS).to(device)
test_noise = torch.randn(4, LATENT_DIM).to(device)
print('Generator output shape:', G(test_noise).shape)  # Expected: (4, 3, 128, 128)
print(f'Generator parameters: {sum(p.numel() for p in G.parameters()):,}')

## Step 5 — DCGAN Discriminator

Architecture: **3×128×128 image → Conv2d blocks → Sigmoid → real/fake probability**

In [ ]:
class Discriminator(nn.Module):
    """
    DCGAN Discriminator
    Input:  RGB image of shape (batch, 3, 128, 128)
    Output: probability in [0, 1] — 1=real, 0=fake
    """
    def __init__(self, fm=64):
        super(Discriminator, self).__init__()

        # Convolutional feature extractor — each block halves spatial size
        self.main = nn.Sequential(
            # 128x128 -> 64x64  (no BatchNorm on first layer per DCGAN paper)
            nn.Conv2d(3, fm, 4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64x64 -> 32x32
            nn.Conv2d(fm, fm * 2, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 -> 16x16
            nn.Conv2d(fm * 2, fm * 4, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 8x8
            nn.Conv2d(fm * 4, fm * 8, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 4x4
            nn.Conv2d(fm * 8, fm * 16, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 16),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Classification head: flatten 4x4 -> single probability
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fm * 16 * 4 * 4, 1),
            nn.Sigmoid(),  # Output probability in [0, 1]
        )
        self._init_weights()

    def forward(self, x):
        features = self.main(x)            # (B, fm*16, 4, 4)
        return self.classifier(features)   # (B, 1)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight.data, 1.0, 0.02)
                nn.init.constant_(m.bias.data, 0)

# Sanity check
D = Discriminator(FEATURE_MAPS).to(device)
test_img = torch.randn(4, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
print('Discriminator output shape:', D(test_img).shape)  # Expected: (4, 1)
print(f'Discriminator parameters: {sum(p.numel() for p in D.parameters()):,}')

## Step 6 — Training Setup

Loss function: **Binary Cross-Entropy (BCE)**  
Optimizer: **Adam** with β₁ = 0.5 (DCGAN paper recommendation)

In [ ]:
# Loss function: Binary Cross-Entropy
criterion = nn.BCELoss()

# Separate Adam optimizers for Generator and Discriminator
optimizer_G = optim.Adam(G.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

# Fixed noise vector — used to visualize progress across epochs
fixed_noise = torch.randn(64, LATENT_DIM, device=device)

# Track losses for plotting
g_losses = []
d_losses = []

print('Training setup complete!')
print(f'  Epochs       : {NUM_EPOCHS}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  LR           : {LEARNING_RATE}')
print(f'  Adam beta1   : {BETA1}')

## Step 7 — Helper: Display Generated Image Grid

In [ ]:
def show_generated_images(images, epoch, save_dir, nrow=8):
    """
    Display and save a grid of generated images.
    images: tensor (N, 3, H, W) in range [-1, 1]
    """
    # Denormalize: [-1, 1] -> [0, 1]
    images = (images * 0.5 + 0.5).clamp(0, 1).detach().cpu()

    # Build image grid
    grid = vutils.make_grid(images, nrow=nrow, padding=2)
    grid_np = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(14, 7))
    plt.imshow(np.clip(grid_np, 0, 1))
    plt.axis('off')
    plt.title(f'Generated Tomato Disease Images — Epoch {epoch}', fontsize=14)

    save_path = os.path.join(save_dir, f'epoch_{epoch:03d}.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.show()
    print(f'  Saved: {save_path}')

print('Helper function defined.')

## Step 8 — Main GAN Training Loop

**Each epoch:**
1. **Train Discriminator**: Learn to distinguish real vs. fake images using BCE loss
2. **Train Generator**: Fool the Discriminator by minimizing BCE(D(G(z)), 1)

⚠️ This may take **30–60 minutes** on Colab GPU or longer on CPU.

In [ ]:
print('=' * 55)
print('  Starting DCGAN Training on Tomato Disease Images')
print('=' * 55)

for epoch in range(1, NUM_EPOCHS + 1):

    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches  = 0

    for batch_idx, (real_images, _) in enumerate(dataloader):

        real_images = real_images.to(device)
        B = real_images.size(0)   # Current batch size

        # Labels: real=1, fake=0
        real_labels = torch.ones(B, 1, device=device)
        fake_labels = torch.zeros(B, 1, device=device)

        # --------------------------------------------------
        # STEP A: Train the Discriminator
        # Goal: maximise log D(real) + log(1 - D(G(z)))
        # --------------------------------------------------
        discriminator = D  # alias for clarity
        discriminator.zero_grad()

        # Real images → Discriminator should output 1
        d_real_output = discriminator(real_images)
        d_loss_real   = criterion(d_real_output, real_labels)

        # Fake images → Discriminator should output 0
        z           = torch.randn(B, LATENT_DIM, device=device)
        fake_images = G(z)
        d_fake_output = discriminator(fake_images.detach())  # detach: don't update G
        d_loss_fake   = criterion(d_fake_output, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_D.step()

        # --------------------------------------------------
        # STEP B: Train the Generator
        # Goal: fool Discriminator → D(G(z)) should be 1
        # --------------------------------------------------
        G.zero_grad()

        # Generator wants D to classify fake images as real
        g_output = discriminator(fake_images)
        g_loss   = criterion(g_output, real_labels)

        g_loss.backward()
        optimizer_G.step()

        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
        num_batches  += 1

    # ---- Average loss over all batches in this epoch ----
    avg_g = epoch_g_loss / num_batches
    avg_d = epoch_d_loss / num_batches
    g_losses.append(avg_g)
    d_losses.append(avg_d)

    print(f'Epoch [{epoch:>3}/{NUM_EPOCHS}]  G Loss: {avg_g:.4f}   D Loss: {avg_d:.4f}')

    # Save and display generated image grid every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        G.eval()
        with torch.no_grad():
            eval_images = G(fixed_noise)
        show_generated_images(eval_images, epoch, SAVE_DIR)
        G.train()

print('\nTraining complete!')

## Step 9 — Save Trained Model Weights

In [ ]:
# Save Generator and Discriminator weights to disk
torch.save(G.state_dict(), os.path.join(MODEL_DIR, 'generator_final.pth'))
torch.save(D.state_dict(), os.path.join(MODEL_DIR, 'discriminator_final.pth'))

print(f'Models saved to: {MODEL_DIR}')
print('  generator_final.pth')
print('  discriminator_final.pth')

## Step 10 — Plot Training Loss Curves

In [ ]:
epochs_range = range(1, len(g_losses) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, g_losses, label='Generator Loss',
         color='#e74c3c', linewidth=2, marker='o', markersize=3)
plt.plot(epochs_range, d_losses, label='Discriminator Loss',
         color='#2980b9', linewidth=2, marker='s', markersize=3)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('GAN Training Loss Curves', fontsize=15)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/outputs/loss_curve.png', dpi=120)
plt.show()
print('Loss curve saved to /content/outputs/loss_curve.png')

## Step 11 — Generate 50 Synthetic Crop Disease Images

Load the trained Generator and generate 50 individual synthetic leaf images.

In [ ]:
SYNTH_DIR = '/content/outputs/synthetic_leaves'
os.makedirs(SYNTH_DIR, exist_ok=True)

# Switch Generator to inference mode
G.eval()

# Generate and save 50 individual synthetic images
for i in range(50):
    z = torch.randn(1, LATENT_DIM, device=device)
    with torch.no_grad():
        img_tensor = G(z).squeeze(0)         # (3, 128, 128)

    # Denormalize: [-1,1] -> [0,1]
    img_tensor = (img_tensor * 0.5 + 0.5).clamp(0, 1)

    # Convert to PIL and save
    pil_img = TF.to_pil_image(img_tensor.cpu())
    pil_img.save(os.path.join(SYNTH_DIR, f'synthetic_leaf_{i + 1}.png'))

print(f'Saved 50 synthetic images to: {SYNTH_DIR}')

# Display a preview grid of all 50 generated images
z_batch = torch.randn(50, LATENT_DIM, device=device)
with torch.no_grad():
    all_synth = G(z_batch)

show_generated_images(all_synth, epoch=999, save_dir=SYNTH_DIR, nrow=10)
print('Done! Synthetic crop disease images are ready for augmentation.')